# Analysis interview

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from matplotlib import pyplot as plt

In [ ]:
pd.set_option('display.float_format','{:,.2f}'.format)
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', None)

### Data

All files cover 2026-09-28 16:30–17:30 UTC, raw from Amberdata. Each file is a dict of DataFrames.

- `data/btc-perp-20260928-small.pkl.xz`: Deribit `BTC-PERPETUAL`, `order-book-events` (20-level snapshots) and `trades`
- `data/eth-perp-20260928-small.pkl.xz`: Deribit `ETH-PERPETUAL`, `order-book-events` and `trades`
- `data/btc-spot-20260928-small.pkl.xz`: Binance `btc_usdt` spot, `trades`

Full days for 2026-09-26 to 2026-09-28 are in the same format, without the `-small` suffix, e.g. `data/btc-perp-20260926.pkl.xz`.

Deribit perp volumes are in USD notional. `exchangeTimestamp` is unix ms.

In [ ]:
btc_perp = pd.read_pickle('data/btc-perp-20260928-small.pkl.xz')
eth_perp = pd.read_pickle('data/eth-perp-20260928-small.pkl.xz')
btc_spot = pd.read_pickle('data/btc-spot-20260928-small.pkl.xz')

## Part 1: BTC perp valuation

We quote BTC options on Deribit and hedge with the BTC perp. We need a bid/ask valuation of the perp, updated on every book and trade event.

1. Convert the raw data into the two packet formats below, in one time-ordered stream. Prices and volumes are `i64`; pick a scaling and say why. Which fields can you not fill from this data?

L2 update packet:

| field | type |
|---|---|
| packet type = 1 | u8 |
| instrument name | str |
| exchange timestamp | i64 ns |
| publish timestamp | i64 ns |
| exchange sequence number | u64 |
| bid count | u8 |
| ask count | u8 |
| five bid slots (price, volume), best first | i64, i64 |
| five ask slots (price, volume), best first | i64, i64 |

Trade update packet:

| field | type |
|---|---|
| packet type = 2 | u8 |
| instrument name | str |
| exchange timestamp | i64 ns |
| publish timestamp | i64 ns |
| exchange sequence number | u64 |
| direction: 1 buy, 2 sell | u8 |
| kind: 1 on-book, 2 block | u8 |
| price | i64 |
| volume | i64 |
| exchange trade id | u64 |

2. Check the data. Are timestamps and sequence numbers ordered? Are there gaps, crossed books, or trades that do not match the book?
3. Build a valuation from the book alone: mid, size-weighted mid, and VWAP to a fixed depth (e.g. $100k each side). Compare them.
4. Trades arrive between book updates. Use them to update the valuation before the next book update. A single aggressive order can fill against many levels, so it can appear as several trades with the same timestamp.
5. How do you measure whether one valuation is better than another? Compare yours on that measure.

## Part 2: Binance spot as a signal

Binance spot often moves before the Deribit perp.

1. Bucket Binance trades by signed notional. What counts as a burst?
2. After a burst, how long until the perp valuation from part 1 moves by at least $X the same way? Show the distribution by burst size.
3. How often is a burst not followed by a perp move, and how often does the perp move with no burst before it?
4. Binance is in Tokyo and we are in London. With 100–120 ms of latency, is the signal still useful?
5. Add the signal to your valuation. Does it improve on your measure from part 1.5?

## Part 3: Inventory and ETH hedge

Trade the BTC perp on the part 2 signal, then hedge the resulting position with the ETH perp.

1. Simulate a strategy that crosses the BTC perp spread on a signal. Track position, PnL and fees (assume 5 bps taker).
2. Estimate the hedge ratio of ETH perp against BTC perp. How stable is it across horizons (1s, 10s, 1min) and over the hour?
3. Build a risk model for the combined BTC/ETH position: what is the remaining risk after the hedge, and how should it limit position size?
4. Skew the BTC perp valuation by inventory, so that a long position lowers our bid and ask. How large should the skew be at maximum position?
5. What would change with more data, or on a day with a larger move?